In [12]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [2]:
def load_and_explore_dataset(file_path):
    print("=" * 60)
    print("LOAD AND EXPLORE DATA")
    print("=" * 60)

    df = pd.read_csv(file_path)
    print("Shape of the dataset:")
    print(df.shape)
    print("\nCheck for missing:")
    print(df.isnull().sum())
    print("\nFirst five rows:")
    print(df.head())
    print("\nDescriptive Statistics:")
    print(df.describe())
    print("\nDataset info:")
    print(df.info())
    print("\nExited Distribution:")
    print(df["Exited"].value_counts())
    print("\nExited percentage Distribution:")
    print(df["Exited"].value_counts(normalize=True))

    return df

In [3]:
def identify_feature(df):
    print("\n" + "=" * 60)
    print("IDENTIFY FEATURES")
    print("=" * 60)

    # feature to drop
    feature_to_drop = [
        "CustomerId",
        "Surname"
    ]

    #numerical columns
    numerical_features = [
        "CreditScore",
        "Age",
        "Tenure",
        "EstimatedSalary",
        "Balance",
        "NumOfProducts",
        "ServiceRating"
    ]

    #categories feature
    categories_features = [
        "Geography",
        "Gender",  
        "HasCrCard",
        "IsActiveMember"
    ]

    print("\nFeature to drop:", feature_to_drop)
    print("\nNumerical features:", numerical_features)
    print("\nCategories features:", categories_features)

    return feature_to_drop, numerical_features, categories_features

In [4]:
def prepare_data(df, numerical_features, categories_features, feature_to_drop):
    print("\n" + "=" * 60)
    print("PREPARE DATA")
    print("=" * 60)

    #drop unrelevant columns
    df_model = df.drop(columns=feature_to_drop)

    #specify label
    # feature to use
    X = df_model.drop("Exited", axis=1)

    #target to use
    y = df_model["Exited"]

    print("\nFeature Shape:", X.shape)
    print("\nTarget Shape:", y.shape)
    print("\nFeature columns list:", list(X.columns))

    return X, y, list(X.columns)

In [5]:
def create_preprocessing_pipeline(numerical_features, categories_features):
    print("\n" + "=" * 60)
    print("CREATE PREPROCESSING PIPELINE")
    print("=" * 60)

    #numerical columns
    numerical_pipeline = Pipeline([
        ('scaler', StandardScaler())
    ])

    #categories features
    categories_pipeline = Pipeline([
        ('onehot', OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])

    #column preprocessing
    preprocessing = ColumnTransformer([
        ('num', numerical_pipeline, numerical_features),
        ('cat', categories_pipeline, categories_features)
    ])

    return preprocessing

In [6]:
def split_data(X, y, test_size=0.2, random_state=42):
    print("\n" + "=" * 60)
    print("SPLITING DATA INTO TRAIN/TEST")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print("\nTrain set size:", X_train.shape[0])
    print("\nTest set size:", X_test.shape[0])
    print("\nTrain churn distribution:", y_train.value_counts())
    print("\nTest churn distribution:", y_test.value_counts())

    return X_train, X_test, y_train, y_test

In [7]:
def create_pipeline(preprocessor):
    print("\n" + "=" * 60)
    print("CREATING PIPELINE MODEL")
    print("=" * 60)

    model_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    print("\nModel pipeline created successfully")

    return model_pipeline

In [8]:
def train_model(model_pipeline, X_train, y_train):
    print("\n" + "=" * 60)
    print(" TRAIN MODEL")
    print("=" * 60)

    model_pipeline.fit(X_train, y_train)

    print("\nModel train successfully")

    #get names steps from preprocessing
    try:
        # num_features = model_pipeline.named_steps["preprocessor"].transformers_[0][2]
        # cat_features = model_pipeline.named_steps["preprocessor"].transformers_[1][2]

        # #get names step from onehotencoder
        # onehot_encoder = model_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
        # cat_feature_name = onehot_encoder.get_feature_names_out(cat_features)

        all_feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()

        #coefficient
        coefficient = model_pipeline.named_steps["classifier"].coef_[0]
        print("\nTop 10 most importance features (by coefficient magnitude)")
        feature_importance = pd.DataFrame({
            "feature": all_feature_names,
            "coefficient": coefficient
        }).sort_values("coefficient", key=abs, ascending=False)
        print(feature_importance.head(10).to_string(index=False))
    except Exception as e:
        print(f"Cannot extract feature name {e}")

    return model_pipeline

In [9]:
def evaluate_model(model_pipeline, X_train, X_test, y_train, y_test):
    print("\n" + "=" * 60)
    print("EVALUATE MODEL")
    print("=" * 60)

    #predicted model
    y_train_pred = model_pipeline.predict(X_train)
    y_test_pred = model_pipeline.predict(X_test)

    #predict model proba
    y_train_prob = model_pipeline.predict_proba(X_train)[:, 1]
    y_test_prob = model_pipeline.predict_proba(X_test)[:, 1]

    #metrics
    y_train_acc_score = accuracy_score(y_train, y_train_pred)
    y_test_acc_score = accuracy_score(y_test, y_test_pred)

    y_train_roc_auc_score = roc_auc_score(y_train, y_train_prob)
    y_test_roc_auc_score = roc_auc_score(y_test, y_test_prob)

    print("\nMETRICS EVALUATION")
    print(f"\nTrain accuracy score: {float(y_train_acc_score):.4f}")
    print(f"\nTest accuracy score: {float(y_test_acc_score):.4f}")
    print(f"\nTrain ROC score: {float(y_train_roc_auc_score):.4f}")
    print(f"\nTest ROC score: {float(y_test_roc_auc_score):.4f}")

    print("\nCLASSIFICATION REPORT OF TRAIN SIZE")
    print("\nClassification report\n", classification_report(y_train, y_train_pred, target_names=["Retained", "Churned"]))

    print("\nCLASSIFICATION REPORT OF TEST SIZE")
    print("\nClassification report\n", classification_report(y_test, y_test_pred, target_names=["Retained", "Churned"]))

    #cross validation
    cv_score = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring="roc_auc")
    print("\nCross validation(5-fold)")
    print(f"  ROC AUC: {cv_score}")
    print(f"  Cross validation mean: {cv_score.mean():.4f}")
    print(f"  Cross validation std: {cv_score.std():4f}")

    metrics = {
        "y_train_acc_score": y_train_acc_score,
        "y_test_acc_score": y_test_acc_score,
        "y_train_roc_auc_score": y_train_roc_auc_score,
        "y_test_roc_auc_score": y_test_roc_auc_score,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "y_train_prob": y_train_prob,
        "y_test_prob": y_test_prob,
        "cv_score": cv_score
    }

    return metrics 

In [13]:
def save_model_artifact(model_pipeline, feature_columns):
    print("\n" + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(model_pipeline, "bank_customer_churned.pkl")
    print("Bank customer churn model saved successfully")

    joblib.dump(feature_columns, "bank_customer_churned_feature_columns.pkl")
    print("Feature columns saved successfully")

    print("\n" + "=" * 60)
    print("ALL MODEL ARTIFACT SAVED")
    print("=" * 60)

In [16]:
def predict_churn(customer_data):
    model_pipeline = joblib.load("bank_customer_churned.pkl")
    feature_columns = joblib.load("bank_customer_churned_feature_columns.pkl")

    customer_df = pd.DataFrame([customer_data])

    for feature in feature_columns:
        if feature not in customer_df.columns:
            raise ValueError(f"Missing require feature {feature}")

    customer_df = customer_df[feature_columns]

    prediction = model_pipeline.predict(customer_df)[0]
    probability = model_pipeline.predict_proba(customer_df)[0]

    result = {
        "prediction": "Churned" if prediction == 1 else "Retained",
        "churn_probability": probability[1],
        "retained_probability": probability[0],
        "risk_level": "High" if probability[1] > 0.7 else "Medium" if probability[0] > 0.4 else "Low"
    }

    return result

In [20]:
def test_predict():

    customer_data = {
        "CreditScore": 400,
        "Geography": "Gammany",
        "Gender": "Female",
        "Age": 45,
        "Tenure": 2,
        "Balance": 100000.0,
        "NumOfProducts": 1,
        "HasCrCard": "No",
        "IsActiveMember": "No",
        "EstimatedSalary": 80000.0,
        "ServiceRating": 1
    }

    customer_data_1 = {
        "CreditScore": 800,
        "Geography": "France",
        "Gender": "Male",
        "Age": 25,
        "Tenure": 4,
        "Balance": 100000.0,
        "NumOfProducts": 1,
        "HasCrCard": "Yes",
        "IsActiveMember": "Yes",
        "EstimatedSalary": 80000.0,
        "ServiceRating": 3
    }


    result = predict_churn(customer_data)
    print("Prediction:", result["prediction"])
    print("Churned Probability:", result["churn_probability"])
    print("Retained Probability:", result["retained_probability"])
    print("Risk Level:", result["risk_level"])


    result_1 = predict_churn(customer_data_1)
    print("Prediction:", result_1["prediction"])
    print("Churned Probability:", result_1["churn_probability"])
    print("Retained Probability:", result_1["retained_probability"])
    print("Risk Level:", result_1["risk_level"])

In [21]:
def main():
    file_path = "cleaned_bank_customer_churn.csv"

    df = load_and_explore_dataset(file_path)

    feature_to_drop, numerical_features, categories_features = identify_feature(df)

    X, y, feature_columns = prepare_data(df, numerical_features, categories_features, feature_to_drop)

    preprocessing = create_preprocessing_pipeline(numerical_features, categories_features)

    X_train, X_test, y_train, y_test = split_data(X, y)

    model_pipeline = create_pipeline(preprocessing)

    model_pipeline = train_model(model_pipeline, X_train, y_train)

    metrics = evaluate_model(model_pipeline, X_train, X_test, y_train, y_test)

    save_model_artifact(model_pipeline, feature_columns)

    test_predict()

In [22]:
if __name__ == "__main__":
    main()

LOAD AND EXPLORE DATA
Shape of the dataset:
(9997, 14)

Check for missing:
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
EstimatedSalary    0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
Exited             0
ServiceRating      0
dtype: int64

First five rows:
   CustomerId   Surname  CreditScore Geography  Gender   Age  Tenure  \
0    15634602  Hargrave          619    France  Female  42.0       2   
1    15647311      Hill          608     Spain  Female  41.0       1   
2    15619304      Onio          502    France  Female  42.0       8   
3    15701354      Boni          699    France  Female  39.0       1   
4    15737888  Mitchell          850     Spain  Female  43.0       2   

   EstimatedSalary    Balance  NumOfProducts HasCrCard IsActiveMember  Exited  \
0        101348.88       0.00              1       Yes            Yes       1   
1    

C:\Users\ola-dev\Desktop\data-science-class-7\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
C:\Users\ola-dev\Desktop\data-science-class-7\venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
